# Inflation Regimes & Market Returns
### *Not all inflation is the same — and your hedge shouldn't be either.*

---

**Central Question:**  
Do traditional inflation hedges (gold, commodities, TIPS, REITs) actually protect investors — and does the answer change depending on the *type* of inflation environment?

**Approach:**
1. Classify 60+ years of US inflation into four distinct regimes
2. Measure real (inflation-adjusted) performance of 7 asset classes within each regime
3. Identify which equity sectors have the highest inflation beta and hedge effectiveness

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.io as pio
sys.path.append('..')
from src import DataFetcher, RegimeClassifier, AssetClassAnalysis, SectorHedgeAnalysis
pio.renderers.default = 'notebook'
pd.set_option('display.float_format', '{:.2f}'.format)
print('Setup complete ✓')

In [ ]:
FRED_API_KEY  = os.environ.get('FRED_API_KEY', 'YOUR_KEY_HERE')
START_DATE    = '1962-01-01'
FORCE_REFRESH = False

## 1. Data

In [ ]:
fetcher = DataFetcher(fred_api_key=FRED_API_KEY, start_date=START_DATE, force_refresh=FORCE_REFRESH)
macro   = fetcher.get_macro()
assets  = fetcher.get_assets()
sectors = fetcher.get_sectors()
master  = fetcher.get_master()
print(f'Macro  : {macro.shape}')
print(f'Master : {master.shape}')

## 2. Regime Classification

In [ ]:
rc            = RegimeClassifier(macro)
macro_regimes = rc.classify_threshold()
print(macro_regimes['regime_label'].value_counts())

In [ ]:
fig = rc.plot_regimes(df=macro_regimes)
fig.show()

## 3. Asset Class Performance by Regime

In [ ]:
master_with_regimes = master.join(macro_regimes[['regime_label', 'regime_id', 'cpi_momentum']], how='left')
aca     = AssetClassAnalysis(master_df=master_with_regimes)
summary = aca.regime_returns_summary()
summary.head(12)

In [ ]:
fig = aca.plot_regime_returns(metric='mean_ret')
fig.show()

In [ ]:
fig = aca.plot_regime_returns(metric='sharpe')
fig.show()

In [ ]:
fig = aca.plot_real_vs_nominal()
fig.show()

In [ ]:
fig = aca.plot_cumulative()
fig.show()

## 4. Sector Hedge Analysis

In [ ]:
sha = SectorHedgeAnalysis(master_df=master_with_regimes)
fig = sha.plot_rolling_correlations(window=24)
fig.show()

In [ ]:
fig = sha.plot_inflation_betas()
fig.show()

In [ ]:
fig = sha.plot_hedge_scorecard()
fig.show()

In [ ]:
fig = sha.plot_sector_regime_heatmap()
fig.show()